<a href="https://colab.research.google.com/github/penajuanmanuel6-hub/automated-etl-pipeline/blob/main/auditoria_economia_videojuegos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3
import pandas as pd

# Crear un entorno de base de datos relacional en memoria para el análisis
conn = sqlite3.connect(':memory:')

# 1. Simulación de tablas del juego (Inventarios de jugadores y catálogo de ítems)
inventario_jugadores = pd.DataFrame({
    'jugador_id': [1001, 1002, 1003, 1004, 1005],
    'item_id': [301, 302, 301, 305, 302],
    'cantidad_poseida': [15, 1, 45, 2, 0],
})

catalogo_items = pd.DataFrame({
    'item_id': [301, 302, 303, 304, 305],
    'nombre_item': ['Poción Vida', 'Espada Legendaria', 'Escudo de Acero', 'Gema Mágica', 'Llave Maestra'],
    'tipo': ['Consumible', 'Arma', 'Armadura', 'Moneda', 'Quest'],
    'valor_tienda': [50, 1200, 450, 5000, 100]
})

# Cargar a tablas SQL locales
inventario_jugadores.to_sql('inventario', conn, index=False, if_exists='replace')
catalogo_items.to_sql('catalogo', conn, index=False, if_exists='replace')

# 2. Consulta SQL analítica limpia (cruce de tablas y agregación)
query_economia = """
    SELECT
        c.nombre_item,
        c.tipo,
        SUM(i.cantidad_poseida) AS total_en_circulacion,
        SUM(i.cantidad_poseida * c.valor_tienda) AS valor_total_mercado
    FROM inventario i
    JOIN catalogo c ON i.item_id = c.item_id
    GROUP BY c.item_id
    ORDER BY valor_total_mercado DESC;
"""

# Ejecución de la consulta analítica y extracción a Pandas
df_reporte_economia = pd.read_sql_query(query_economia, conn)
conn.close()

print("--- REPORTE DE BALANCE ECONÓMICO IN-GAME ---")
display(df_reporte_economia)

--- REPORTE DE BALANCE ECONÓMICO IN-GAME ---


,nombre_item,tipo,total_en_circulacion,valor_total_mercado
0,Poción Vida,Consumible,60,3000
1,Espada Legendaria,Arma,1,1200
2,Llave Maestra,Quest,2,200
